# 2.2 — Functions, errors, and testing

Turn the record processing from 2.1 into named, reusable functions whose inputs and results can be checked.

## Introduction

Use this Notebook to verify the lesson ideas with executable code.

## Learning outcomes

- Define and call functions with parameters and useful return values.
- Explain local scope, defaults, keyword arguments, docstrings, and type hints as parts of a contract.
- Separate validation, calculation, state change, and presentation responsibilities.
- Classify syntax, runtime, and logical errors and keep exception handling narrow.
- Test return values, raised exceptions, and state before and after a call.

> **Learning route:** Required: 2.2.1–2.2.6  |  Integration: 2.2.7


## 2.2.1 Define a function with explicit inputs and output

Define a function with `def`. Defining it does not run its body; calling it runs the indented body. The definition must have executed before the call.

In [ ]:
def completion_rate(completed, registered):
    return completed / registered * 100

rate = completion_rate(32, 40)
print(f"Completion rate: {rate:.1f}%")


### Parameters are inputs and return values are outputs

`completed` and `registered` in the definition are parameters; `32` and `40` in the call are arguments. `return` sends a result to the caller and ends the function. `print()` only displays text, so it cannot replace a returned value needed by later calculations.

In [ ]:
def add_with_print(a, b):
    print(a + b)

def add_with_return(a, b):
    return a + b

printed_result = add_with_print(2, 3)
returned_result = add_with_return(2, 3)
print("Print version returned:", printed_result)
print("Returned value reused:", returned_result * 2)


## 2.2.2 Control scope and communicate the calling contract

A variable assigned inside a function is local and normally cannot be read outside it. Receive required values as parameters and return results. Reducing dependence on external state makes behaviour easier to repeat and test.

In [ ]:
def difference(planned, actual):
    gap = planned - actual
    return gap

print(difference(40, 34))
# print(gap)  # NameError: gap is local to the function


### Defaults and keyword arguments clarify calls

Give optional parameters default values, placing required parameters first. Keyword arguments make calls readable when several values have the same type.

In [ ]:
def format_centre(name, completed, registered, decimals=1):
    rate = completed / registered * 100
    return f"{name}: {rate:.{decimals}f}%"

print(format_centre("North", completed=32, registered=40))
print(format_centre("South", completed=24, registered=35, decimals=2))


### A docstring and type hints communicate the contract

Describe purpose, inputs, return value, and invalid-input policy in the docstring. Type hints help readers and tools; Python does not automatically enforce them at runtime.

In [ ]:
def safe_rate(completed: int, registered: int) -> float | None:
    """Return completion percentage, or None when registration is not positive."""
    if registered <= 0:
        return None
    return completed / registered * 100

print(safe_rate(32, 40))
print(safe_rate(0, 0))


## 2.2.3 Separate validation, processing, and presentation

Separate record validation, rate calculation, and display instead of mixing them into one long process. Distinguish a missing required key from a value outside its valid range.

In [ ]:
REQUIRED_FIELDS = {"name", "registered", "completed"}

def validate_centre(centre):
    missing = REQUIRED_FIELDS - centre.keys()
    if missing:
        raise KeyError(f"Missing required fields: {sorted(missing)}")
    if centre["registered"] < 0 or centre["completed"] < 0:
        raise ValueError("Counts cannot be negative")
    if centre["completed"] > centre["registered"]:
        raise ValueError("Completed cannot exceed registered")

def centre_rate(centre):
    validate_centre(centre)
    if centre["registered"] == 0:
        return None
    return centre["completed"] / centre["registered"] * 100

centre = {"name": "North", "registered": 40, "completed": 32}
print(centre_rate(centre))


## 2.2.4 Read errors and catch only expected exceptions

A syntax error prevents Python from parsing the program. A runtime error raises an exception during execution. A logic error runs but produces the wrong result. Read a traceback from its final exception name and message, then move upward to the relevant line in your code.

In [ ]:
def broken_rate(completed, registered):
    return completed / registerd * 100  # Intentional spelling error

try:
    broken_rate(32, 40)
except NameError as error:
    print(type(error).__name__)
    print(error)


### Catch only expected exceptions, in a narrow region

Place only the operation that may fail in `try` and catch a specific exception you can handle. `else` runs when there was no exception; `finally` supports cleanup required in either case. Avoid `except Exception: pass`, which hides causes.

In [ ]:
raw = "40"
try:
    registered = int(raw)
except ValueError:
    print("Not a valid integer")
else:
    print("Registered:", registered)
finally:
    print("Input check finished")


## 2.2.5 Test normal, boundary, and invalid cases

A normal case alone cannot expose boundary bugs. State expected results with `assert`, including ordinary values, a boundary such as zero, and invalid values. Compare floats with a tolerance. Assertions are useful checks here, but do not replace validation of user data.

In [ ]:
assert abs(safe_rate(32, 40) - 80.0) < 0.0001
assert safe_rate(0, 0) is None
assert safe_rate(1, -1) is None

try:
    validate_centre({"name": "Bad", "registered": 5, "completed": 7})
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

print("All tests passed")


## 2.2.6 Specify and test search and mutation contracts

Use the 2.1 equipment register and implement `find_asset(assets, asset_id)`, `add_asset(...)`, `mark_available(...)`, and `remove_asset(...)`. Absence from search returns `None`; blank or duplicate additions raise `ValueError`; absent update or removal targets raise `KeyError`. Test normal and invalid cases plus count and order before and after each mutation.


In [ ]:
def find_book(books, book_id):
    for book in books:
        if book["id"] == book_id:
            return book
    return None

def mark_as_read(books, book_id):
    book = find_book(books, book_id)
    if book is None:
        raise KeyError(book_id)
    book["read"] = True
    return book

books = [{"id": "B001", "title": "Python Basics", "read": False}]
changed = mark_as_read(books, "B001")
print(changed)
print(books)


### Separate an invalid new value from an absent update target

A blank or duplicate ID violates the rule for a newly supplied value, so raise `ValueError`. An update or removal requested for an ID that is not stored has a missing target, so raise `KeyError`. Specific exceptions let an automatic checker verify the cause, not only that something failed.

In [ ]:
def add_book(books, book_id, title):
    clean_id = book_id.strip()
    clean_title = title.strip()
    if not clean_id or not clean_title:
        raise ValueError("id and title are required")
    if find_book(books, clean_id) is not None:
        raise ValueError(f"duplicate id: {clean_id}")
    book = {"id": clean_id, "title": clean_title, "read": False}
    books.append(book)
    return book


### Test state before and after a call, not only its return value

For a mutating function, checking only the returned dictionary is insufficient. Check that count increased once, the returned dictionary is the stored object, and existing order was preserved. For a calculation-only function, test the opposite contract: its input remains unchanged.

In [ ]:
before_count = len(books)
added = add_book(books, " B002 ", " Working with Data ")
assert len(books) == before_count + 1
assert added is books[-1]
assert added == {"id": "B002", "title": "Working with Data", "read": False}

try:
    add_book(books, "B002", "Duplicate")
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

print("STATE TESTS PASSED")


### A supplied checker is another program that consumes the contract

You do not need to author or fully understand the checker yet. It imports `library_manager.py` and calls the named functions with normal, boundary, and invalid cases. Keep filename, function names, parameters, returns, mutations, and exceptions compatible. When a check reports `NG`, change only your program.

## 2.2.7 Integrated practice: connect tested functions

Using the three-centre list from 2.1, separate the work into `validate_centre()`, `centre_rate()`, and `summarise_centres()`. The last function returns a dictionary containing names below 75%, the set of districts, total registration, and total completion. Test three valid records, zero registration, a missing required key, and completion above registration.

In [ ]:
# Write the transfer solution here.


## Summary

- Used functions to give each processing responsibility a stable name and contract.
- Handled only predictable exceptional cases near the operation that can fail.
- Verified normal, boundary, invalid, and state-changing behaviour independently.

## Next

Lesson 2.3 uses these function and error contracts to read persistent records from CSV and save a separately verifiable result.

**Estimated learning time:** about 3 hours
